# Etape 1 - reprendre ta spécification

Catégorie choisie : "Fantasy"

Etape 1 - le périmètre

Nombre de livres : 48
Nombre de pages : 3

Etape 2 - le bloc qui se répète

document.querySelectorAll('article.product_pod').length
20

Etape 3 - le tableau des cinq données

Donnée                              Sélecteur                                   Contenu ou attribut ?
titre                               h3 a                                        Attribut title
prix                                p.price_color                               Contenu
disponibilité                       p.instock                                   Contenu
note                                p.star-rating                               Attribut (dans le nom de l'attribut)
lien vers la fiche                  h3 a                                        Attribut href

Etape 4 - deux sélecteurs testés

document.querySelectorAll('article.product_pod p.price_color').length
20

Etape 5 - la page suivante

Sélecteur du lien "next" : li.next a
href : page-2.html

Etape 6 - le contrôle du code source

En copiant le code de l'élément, je le retrouve dans le code source de la page Web.
Le site est statique. Le prix est directement inscrit dans le fichier HTML.
L'information serait donc récupérable par un script qui téléchargerait cette page.

Etape 7 - deux données de la fiche détaillée

Quantité en stock : contenu de p.instock
Description du produit : contenu de p

In [70]:
# Etape 2 - une page, pour valider la chaine

import requests

URL = "https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (bourreauclemence@gmail.com)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status   :", r.status_code)
print("longueur :", len(r.text), "caractères")
print("début    :", r.text[:60])

status   : 200
longueur : 51222 caractères
début    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [71]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvées :", len(fiches))

fiches trouvées : 20


In [72]:
fiche = fiches[0]

prix = fiche.select_one("p.price_color").get_text(strip=True)

lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix     :", prix)
print("titre    :", titre)
print("href     :", href)

prix     : £18.78
titre    : Unicorn Tracks
href     : ../../../unicorn-tracks_951/index.html


In [73]:
classes = fiche.select_one("p.star-rating")["class"]

NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

note = NOTES[classes[1]]
print("note :", note)

note : 3


In [74]:
from urllib.parse import urljoin

print(urljoin(URL, href))

https://books.toscrape.com/catalogue/unicorn-tracks_951/index.html


In [75]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }

lire_fiche(fiches[0], URL)

{'titre': 'Unicorn Tracks',
 'prix': '£18.78',
 'dispo': 'In stock',
 'note': 3,
 'url': 'https://books.toscrape.com/catalogue/unicorn-tracks_951/index.html'}

In [76]:
import pandas as pd

donnees = [lire_fiche(fiche, URL) for fiche in fiches]

df_page1 = pd.DataFrame(donnees)

df_page1

,titre,prix,dispo,note,url
0,Unicorn Tracks,£18.78,In stock,3,https://books.toscrape.com/catalogue/unicorn-t...
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",£25.02,In stock,3,https://books.toscrape.com/catalogue/saga-volu...
2,Princess Between Worlds (Wide-Awake Princess #5),£13.34,In stock,5,https://books.toscrape.com/catalogue/princess-...
3,Masks and Shadows,£56.40,In stock,2,https://books.toscrape.com/catalogue/masks-and...
4,Crown of Midnight (Throne of Glass #2),£43.29,In stock,3,https://books.toscrape.com/catalogue/crown-of-...
5,"Avatar: The Last Airbender: Smoke and Shadow, ...",£28.09,In stock,2,https://books.toscrape.com/catalogue/avatar-th...
6,A Court of Thorns and Roses (A Court of Thorns...,£52.37,In stock,1,https://books.toscrape.com/catalogue/a-court-o...
7,Throne of Glass (Throne of Glass #1),£35.07,In stock,3,https://books.toscrape.com/catalogue/throne-of...
8,The Glittering Court (The Glittering Court #1),£44.28,In stock,1,https://books.toscrape.com/catalogue/the-glitt...
9,Hollow City (Miss Peregrine’s Peculiar Childre...,£42.98,In stock,1,https://books.toscrape.com/catalogue/hollow-ci...


In [77]:
# Etape 3 - toutes les pages de liste
suivant = soup.select_one("li.next a")

import time

def collecter_toutes_les_pages(url_depart, pages_max=5):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"    page {page} : status {r.status_code}, on s'arrête")
            break
    
        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"    page {page} : {len(fiches)} livres (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)
    return livres

livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html")
print("collecte :", len(livres), "livres")

    page 1 : 20 livres (total 20)
    page 2 : 20 livres (total 40)
    page 3 : 8 livres (total 48)
collecte : 48 livres


In [78]:
df_livres = pd.DataFrame(livres)

df_livres.head()

,titre,prix,dispo,note,url
0,Unicorn Tracks,£18.78,In stock,3,https://books.toscrape.com/catalogue/unicorn-t...
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",£25.02,In stock,3,https://books.toscrape.com/catalogue/saga-volu...
2,Princess Between Worlds (Wide-Awake Princess #5),£13.34,In stock,5,https://books.toscrape.com/catalogue/princess-...
3,Masks and Shadows,£56.40,In stock,2,https://books.toscrape.com/catalogue/masks-and...
4,Crown of Midnight (Throne of Glass #2),£43.29,In stock,3,https://books.toscrape.com/catalogue/crown-of-...


In [79]:
df_livres["prix_gbp"] = df_livres["prix"].str.replace("£", "", regex=False).astype(float)

df_livres.head()

,titre,prix,dispo,note,url,prix_gbp
0,Unicorn Tracks,£18.78,In stock,3,https://books.toscrape.com/catalogue/unicorn-t...,18.78
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",£25.02,In stock,3,https://books.toscrape.com/catalogue/saga-volu...,25.02
2,Princess Between Worlds (Wide-Awake Princess #5),£13.34,In stock,5,https://books.toscrape.com/catalogue/princess-...,13.34
3,Masks and Shadows,£56.40,In stock,2,https://books.toscrape.com/catalogue/masks-and...,56.40
4,Crown of Midnight (Throne of Glass #2),£43.29,In stock,3,https://books.toscrape.com/catalogue/crown-of-...,43.29


In [80]:
# Etape 4 - le second niveau
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}
    
    soup = BeautifulSoup(r.text, "html.parser")

    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)
    
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    time.sleep(1)

    return {
        "url": url,
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }

url_test = "https://books.toscrape.com/catalogue/unicorn-tracks_951/index.html"
print(lire_detail(url_test))

{'url': 'https://books.toscrape.com/catalogue/unicorn-tracks_951/index.html', 'categorie': 'Fantasy', 'stock': 'In stock (16 available)', 'upc': '7ae099f3898e0209'}


In [81]:
details = df_livres['url'].apply(lire_detail)

df_details = pd.DataFrame(list(details))

df_details.head()

,url,categorie,stock,upc
0,https://books.toscrape.com/catalogue/unicorn-t...,Fantasy,In stock (16 available),7ae099f3898e0209
1,https://books.toscrape.com/catalogue/saga-volu...,Fantasy,In stock (16 available),c849f0b2f5d6a742
2,https://books.toscrape.com/catalogue/princess-...,Fantasy,In stock (16 available),0e691eda369f4e09
3,https://books.toscrape.com/catalogue/masks-and...,Fantasy,In stock (16 available),fa9610a50a1bf149
4,https://books.toscrape.com/catalogue/crown-of-...,Fantasy,In stock (16 available),7267864328ae8b9d


In [82]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0

df_details["nb_stock"] = df_details["stock"].apply(nombre_en_stock)

df_details.head()

,url,categorie,stock,upc,nb_stock
0,https://books.toscrape.com/catalogue/unicorn-t...,Fantasy,In stock (16 available),7ae099f3898e0209,16
1,https://books.toscrape.com/catalogue/saga-volu...,Fantasy,In stock (16 available),c849f0b2f5d6a742,16
2,https://books.toscrape.com/catalogue/princess-...,Fantasy,In stock (16 available),0e691eda369f4e09,16
3,https://books.toscrape.com/catalogue/masks-and...,Fantasy,In stock (16 available),fa9610a50a1bf149,16
4,https://books.toscrape.com/catalogue/crown-of-...,Fantasy,In stock (16 available),7267864328ae8b9d,16


In [83]:
df_livres.head()

,titre,prix,dispo,note,url,prix_gbp
0,Unicorn Tracks,£18.78,In stock,3,https://books.toscrape.com/catalogue/unicorn-t...,18.78
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",£25.02,In stock,3,https://books.toscrape.com/catalogue/saga-volu...,25.02
2,Princess Between Worlds (Wide-Awake Princess #5),£13.34,In stock,5,https://books.toscrape.com/catalogue/princess-...,13.34
3,Masks and Shadows,£56.40,In stock,2,https://books.toscrape.com/catalogue/masks-and...,56.40
4,Crown of Midnight (Throne of Glass #2),£43.29,In stock,3,https://books.toscrape.com/catalogue/crown-of-...,43.29


In [84]:
# Etape 5 - assembler et nettoyer
colonnes_finales = ["titre", "categorie", "prix_gbp", "note", "nb_stock", "upc", "url"]

df_final = pd.merge(
    left=df_livres[["titre", "note", "url", "prix_gbp"]],
    right=df_details[["url", "categorie", "upc", "nb_stock"]],
    on="url",
    how="inner"
)[colonnes_finales]

df_final.head()

,titre,categorie,prix_gbp,note,nb_stock,upc,url
0,Unicorn Tracks,Fantasy,18.78,3,16,7ae099f3898e0209,https://books.toscrape.com/catalogue/unicorn-t...
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",Fantasy,25.02,3,16,c849f0b2f5d6a742,https://books.toscrape.com/catalogue/saga-volu...
2,Princess Between Worlds (Wide-Awake Princess #5),Fantasy,13.34,5,16,0e691eda369f4e09,https://books.toscrape.com/catalogue/princess-...
3,Masks and Shadows,Fantasy,56.40,2,16,fa9610a50a1bf149,https://books.toscrape.com/catalogue/masks-and...
4,Crown of Midnight (Throne of Glass #2),Fantasy,43.29,3,16,7267864328ae8b9d,https://books.toscrape.com/catalogue/crown-of-...


In [85]:
# Etape 6 - analyser

print(f"Nombre de livres : {len(df_final)}")
print(f"Prix moyen : {round(df_final["prix_gbp"].mean(), 2)} £")

cher = df_final.loc[df_final["prix_gbp"].idxmax()]
print(f"Livre le plus cher : {cher["titre"]}, avec un prix de {cher["prix_gbp"]} £")

print(f" Répartition des notes de 1 à 5 : {df_final["note"].value_counts()}")

print(f"Nombre de valeurs dans categorie : {df_final["categorie"].nunique()}")
print(f"Si le résultat avait été différent de 1, cela aurait voulu dire qu'il ya a eu une erreur dans le processus et que l'export des données n'a pas été fait sur une seule catégorie mais sur plusieurs.")

Nombre de livres : 48
Prix moyen : 39.59 £
Livre le plus cher : Myriad (Prentor #1), avec un prix de 58.75 £
 Répartition des notes de 1 à 5 : note
4    11
5    10
3     9
2     9
1     9
Name: count, dtype: int64
Nombre de valeurs dans categorie : 1
Si le résultat avait été différent de 1, cela aurait voulu dire qu'il ya a eu une erreur dans le processus et que l'export des données n'a pas été fait sur une seule catégorie mais sur plusieurs.


In [86]:
# Etape 7 - livrer
df_final.to_csv("../data/liste_livres_fantasy.csv", sep=";", encoding="utf-8", index=False)